## Deploying the Streamlit App with ngrok

To make your Streamlit app accessible publicly, we'll use `ngrok`. This will create a secure tunnel to your local Streamlit instance.

### Setup: Set your ngrok auth token

**Option A: Set it in PowerShell (temporary - for current session):**
```powershell
$env:NGROK_AUTH_TOKEN = "34aRtaj0nr65AUUcSXRVhrja2td_qxWTdUu73zV1gwSe4tGh"
```

**Option B: Set it permanently (recommended):**
```powershell
[System.Environment]::SetEnvironmentVariable('NGROK_AUTH_TOKEN', '34aRtaj0nr65AUUcSXRVhrja2td_qxWTdUu73zV1gwSe4tGh', 'User')
```
*Note: You'll need to restart your terminal/notebook after setting it permanently.*


### Option 1: Run Streamlit locally (recommended)

In PowerShell Terminal, run:
```powershell
pip install -r requirements.txt
streamlit run app.py
```

Then open the local URL shown in the terminal (usually http://localhost:8501).


### Option 2: Run Streamlit with ngrok from this notebook

**First, run the cell below to set your ngrok token (if not set already), then execute the ngrok tunnel cell.**


### About the Stock Price Prediction App

The full Streamlit application (`app.py`) is located in the workspace root directory.

**Features:**
- **Model Selection**: Auto-selects RandomForest or XGBoost based on data history length
- **Caching**: Trained models are saved/loaded from disk to avoid retraining
- **Hyperparameter Search**: Optional RandomizedSearchCV for tuning
- **Evaluation Metrics**: RMSE, MAE, and R² on walk-forward validation
- **Explainability**: Feature importances and SHAP values (optional)
- **Visualization**: Plotly charts showing historical data + 5-day predictions
- **Backtesting**: Chronological holdout testing with diagnostics


In [ ]:
# Set ngrok auth token for this session
import os
NGROK_TOKEN = "34aRtaj0nr65AUUcSXRVhrja2td_qxWTdUu73zV1gwSe4tGh"
os.environ['NGROK_AUTH_TOKEN'] = NGROK_TOKEN
print("✅ Ngrok auth token set for this notebook session")

%pip install pyngrok streamlit joblib yfinance scikit-learn plotly -q

import subprocess
import time
from pyngrok import ngrok
import os
import sys

# Start Streamlit app in the background
print("Starting Streamlit app in background...")
process = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "True"],
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE)

time.sleep(5)

try:
    token = os.environ.get('NGROK_AUTH_TOKEN')
    if token:
        ngrok.set_auth_token(token)
        print('✅ ngrok auth token set from environment')
    else:
        print('⚠️ No NGROK_AUTH_TOKEN found in environment — public URL may be limited.')
        print('Please run the previous cell to set your ngrok token.')
    
    public_url = ngrok.connect(8501)
    print(f"\n🌐 Streamlit app is running at: {public_url}")
    print("To stop the app, interrupt this cell's execution.")

    while process.poll() is None:
        stdout_line = process.stdout.readline().decode('utf-8').strip()
        stderr_line = process.stderr.readline().decode('utf-8').strip()
        if stdout_line:
            print('Streamlit (stdout):', stdout_line)
        if stderr_line:
            print('Streamlit (stderr):', stderr_line)
        time.sleep(1)

    stdout, stderr = process.communicate()
    if stdout:
        print('Streamlit (final stdout):', stdout.decode('utf-8'))
    if stderr:
        print('Streamlit (final stderr):', stderr.decode('utf-8'))
    ngrok.kill()
except Exception as e:
    print('❌ Error creating ngrok tunnel or during Streamlit app execution:', e)
    if process.poll() is not None:
        stdout, stderr = process.communicate()
        print('Streamlit stdout:', stdout.decode('utf-8'))
        print('Streamlit stderr:', stderr.decode('utf-8'))
    ngrok.kill()

